<a href="https://colab.research.google.com/github/silvatria/pembelajaran-mesin-254107023001/blob/main/JS09_TL01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install faiss-cpu annoy hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 9.3 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 46.7 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520034 sha256=d1d12350fd19d33a550f42a9dbbb6b265867be6d902d2482935df49a1e7cc691
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679801 sha256=4dc29dca88dc877677d13d5d04eda7752d09c4e21d1aba1513bdb0b786796090
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

In [ ]:
import time
import numpy as np
from annoy import AnnoyIndex
import faiss
import hnswlib
import pandas as pd

# Konfigurasi parameter eksperimen
configurations = [
    {"num_data": 1000, "dim": 2, "metric": "euclidean"},
    {"num_data": 1000, "dim": 2, "metric": "cosine"},
    {"num_data": 1000000, "dim": 5, "metric": "euclidean"},
    {"num_data": 1000000, "dim": 5, "metric": "cosine"},
]

results = []

for config in configurations:
    n = config["num_data"]
    d = config["dim"]
    metric = config["metric"]

    print(f"\n--- Menjalankan eksperimen: {n} data / {d}D / Metric: {metric} ---")

    # 1. Generate dataset acak
    np.random.seed(42)
    data = np.random.random((n, d)).astype('float32')

    # Normalisasi data khusus untuk metric cosine
    if metric == "cosine":
        faiss.normalize_L2(data)

    query_vector = np.random.random((1, d)).astype('float32')
    if metric == "cosine":
        faiss.normalize_L2(query_vector)

    # ==================== 1. ANNOY ====================
    annoy_metric = 'angular' if metric == 'cosine' else 'euclidean'
    t_start = time.time()

    annoy_index = AnnoyIndex(d, annoy_metric)
    for i, vector in enumerate(data):
        annoy_index.add_item(i, vector)
    annoy_index.build(n_trees=10) # 10 trees

    t_index_annoy = time.time() - t_start

    t_start = time.time()
    _ = annoy_index.get_nns_by_vector(query_vector[0], 10)
    t_search_annoy = time.time() - t_start

    # ==================== 2. FAISS ====================
    t_start = time.time()

    if metric == 'cosine':
        # FAISS inner product setelah normalisasi setara dengan cosine similarity
        faiss_index = faiss.IndexFlatIP(d)
    else:
        faiss_index = faiss.IndexFlatL2(d)

    faiss_index.add(data)
    t_index_faiss = time.time() - t_start

    t_start = time.time()
    _, _ = faiss_index.search(query_vector, 10)
    t_search_faiss = time.time() - t_start

    # ==================== 3. HNSWLIB ====================
    hnsw_metric = 'cosine' if metric == 'cosine' else 'l2'
    t_start = time.time()

    hnsw_index = hnswlib.Index(space=hnsw_metric, dim=d)
    hnsw_index.init_index(max_elements=n, ef_construction=200, M=16)
    hnsw_index.add_items(data, np.arange(n))
    hnsw_index.set_ef(50)

    t_index_hnsw = time.time() - t_start

    t_start = time.time()
    _, _ = hnsw_index.knn_query(query_vector, k=10)
    t_search_hnsw = time.time() - t_start

    # Simpan hasil
    results.append({
        "Jumlah Data/Dimensi": f"{n:,} / {d}D ({metric})",
        "ANNOY (Index/Search)": f"{t_index_annoy:.4f}s / {t_search_annoy:.6f}s",
        "FAISS (Index/Search)": f"{t_index_faiss:.4f}s / {t_search_faiss:.6f}s",
        "HNSW (Index/Search)": f"{t_index_hnsw:.4f}s / {t_search_hnsw:.6f}s"
    })

# Tampilkan dalam bentuk DataFrame / Tabel
df_result = pd.DataFrame(results)
print("\n=== HASIL AKHIR BENCHMARK ===")
print(df_result.to_markdown(index=False))

# Opsional: Export ke CSV
df_result.to_csv("hasil_benchmark_lab1.csv", index=False)

# === HASIL AKHIR BENCHMARK ===
# | Jumlah Data/Dimensi        | ANNOY (Index/Search)   | FAISS (Index/Search)   | HNSW (Index/Search)   |
# |:---------------------------|:-----------------------|:-----------------------|:----------------------|
# | 1,000 / 2D (euclidean)     | 0.0228s / 0.000079s    | 0.0001s / 0.000078s    | 0.0544s / 0.000056s   |
# | 1,000 / 2D (cosine)        | 0.0317s / 0.000057s    | 0.0001s / 0.000053s    | 0.0455s / 0.000059s   |
# | 1,000,000 / 5D (euclidean) | 19.6552s / 0.000097s   | 0.0138s / 0.012874s    | 172.4912s / 0.000102s |
# | 1,000,000 / 5D (cosine)    | 28.4148s / 0.000102s   | 0.0147s / 0.004500s    | 159.4521s / 0.000088s |


--- Menjalankan eksperimen: 1000 data / 2D / Metric: euclidean ---

--- Menjalankan eksperimen: 1000 data / 2D / Metric: cosine ---

--- Menjalankan eksperimen: 1000000 data / 5D / Metric: euclidean ---
